> Processing of the results from training (including *clusterization*)

In [1]:
import pandas as pd

import trapallada.analysis
import trapallada.results.analysis

So that `import`ed Python modules are automatically reloaded when modified

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


The base directory

In [3]:
base_dir = trapallada.analysis.get_base_directory()
base_dir

PosixPath('/home/manu/Sync/git/trapallada')

In [4]:
results_dir = base_dir / 'results' / 'multichannel' / 'rates_large_infs' / 'fft_z_30'
assert results_dir.exists()

From the results directory,
- the `Dataset`,
- the headers of the corresponding files,
- the ML model,
- a `DataFrame` with results from training, 
- a `dict` with the parameters

In [5]:
ds, positions_df, model, training_df, parameters = trapallada.results.analysis.load_dir(results_dir)

trapallada.data - INFO - computing the FFT of each signal...
trapallada.data - INFO - dismissing the right half of each FFT (length is now 113)...


Clustering using [HDBSCAN](https://github.com/scikit-learn-contrib/hdbscan) with pretty much every default parameter. See [this notebook](https://nbviewer.org/github/scikit-learn-contrib/hdbscan/blob/master/notebooks/How%20HDBSCAN%20Works.ipynb) for an explanation.

In [6]:
clustered_df = trapallada.results.analysis.cluster(ds, model.encoder) # type: ignore

/home/manu/Sync/git/trapallada/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/manu/Sync/git/trapallada/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/manu/Sync/git/trapallada/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/manu/Sync/git/trapallada/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Position information extracted from the *headers* of the FITs files is merged with the above `DataFrame`

In [7]:
res_df = trapallada.results.analysis.augment_with_positions(clustered_df, positions_df)

Some *alleged* labels associated with spatial coordinates from SIMBAD (XMATCH) database are appended

In [8]:
catalog_file = base_dir / 'experiments' / 'Cross_Simbad_r3.5.csv'
    
res_df = trapallada.results.analysis.augment_with_simbad(res_df, catalog_file)

/home/manu/Sync/git/trapallada/trapallada/results/analysis.py:120: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  catalog_df = pd.read_csv(catalog_file)


Two new columns are added
- one just combines the *tar* and *FIT* file, which univocally determines the signal,
- the other yields the number of window *within* every (*tar* file, *FIT* file) combination

In [9]:
res_df = trapallada.results.analysis.augment_with_window_and_signal_ids(res_df)